In [1]:
#!pip install pytorch_lightning==1.6.4
#!pip install pytorch_forecasting==0.10.1
#!pip install torchmetrics==0.5.0

In [2]:
#!pip install torch==1.13.1+cu117 torchvision==0.14.1+cu117 torchaudio==0.13.1 --extra-index-url https://download.pytorch.org/whl/cu117

In [1]:
# Generic libraries
import pandas as pd
import seaborn as sns
import numpy as np
# Pytorch / Pytorch lightning dependencies
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger
import torch
# Pytorch Forecasting library
from pytorch_forecasting import Baseline, TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import TorchNormalizer, GroupNormalizer, MultiNormalizer
from pytorch_forecasting.metrics import SMAPE, MAE,RMSE, PoissonLoss, QuantileLoss, MultiLoss
from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters

2023-07-09 22:18:39.463284: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
%cd tft

/home/usuario/Descargas/TFTrevisar/tftdef/tft


In [3]:
# Making sure all needed libraries of Google lib and exact versions are matched
#!pip install -r requirements.txt --q
!pwd

/home/usuario/Descargas/TFTrevisar/tftdef/tft


In [5]:
trayectorias_df = pd.read_csv("../data/trayectorias/data.csv", encoding='utf8', index_col=0)
trayectorias_df = trayectorias_df.rename({'aerodromeOfDeparture':'ruta'},axis=1)
trayectorias_df

/tmp/ipykernel_162013/1386230639.py:1: DtypeWarning: Columns (38) have mixed types. Specify dtype option on import or set low_memory=False.
  trayectorias_df = pd.read_csv("../data/trayectorias/data.csv", encoding='utf8', index_col=0)


,level_0,Unnamed: 0,index,fpId,icao24,callsign,latitude,longitude,speed,vspeed,...,min_temp,min_temp_timestamp,validity_from,validity_to,timespan,day_of_week,hav_distance,time_of_day,is_outlier,sector
0,0,0,0,AT05414795,40697C,BAW456,51.477402,-0.4745,141.0,2689.0,...,1,2022-01-01 05:00:00,2022-01-01 05:00:00,2022-01-02 11:00:00,30,5,773.208995,night,False,1
1,1,1,1,AT05414795,40697C,BAW456,51.477200,-0.4900,132.0,3264.0,...,1,2022-01-01 05:00:00,2022-01-01 05:00:00,2022-01-02 11:00:00,30,5,773.054840,night,False,1
2,2,2,2,AT05414795,40697C,BAW456,51.477100,-0.5048,137.0,2175.0,...,1,2022-01-01 05:00:00,2022-01-01 05:00:00,2022-01-02 11:00:00,30,5,772.914619,night,False,1
3,3,3,3,AT05414795,40697C,BAW456,51.476898,-0.5255,158.0,1984.0,...,1,2022-01-01 05:00:00,2022-01-01 05:00:00,2022-01-02 11:00:00,30,5,772.715249,night,False,1
4,4,4,4,AT05414795,40697C,BAW456,51.475601,-0.5393,168.0,1984.0,...,1,2022-01-01 05:00:00,2022-01-01 05:00:00,2022-01-02 11:00:00,30,5,772.504437,night,False,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2971103,446637,446637,59719,AT05829591,34644C,IBS3707,40.447399,-3.5096,145.0,-640.0,...,8,2022-10-01 04:00:00,2022-09-30 16:00:00,2022-10-01 22:00:00,30,4,4.181025,evening,False,1
2971104,446638,446638,59720,AT05829591,34644C,IBS3707,40.454498,-3.5168,149.0,-833.0,...,8,2022-10-01 04:00:00,2022-09-30 16:00:00,2022-10-01 22:00:00,30,4,3.563596,evening,False,1
2971105,446639,446639,59721,AT05829591,34644C,IBS3707,40.463299,-3.5258,148.0,-640.0,...,8,2022-10-01 04:00:00,2022-09-30 16:00:00,2022-10-01 22:00:00,30,4,2.796858,evening,False,1
2971106,446640,446640,59722,AT05829591,34644C,IBS3707,40.471802,-3.5344,152.0,-577.0,...,8,2022-10-01 04:00:00,2022-09-30 16:00:00,2022-10-01 22:00:00,30,4,2.062430,evening,False,1


In [6]:
#Escalo los valores
from sklearn.preprocessing import MinMaxScaler

scaler1 = MinMaxScaler(feature_range=(0,1),copy=True)
scaler2 = MinMaxScaler(feature_range=(0,1),copy=True)
scaler3 = MinMaxScaler(feature_range=(0,1),copy=True)
scaler4 = MinMaxScaler(feature_range=(0,1),copy=True)
scaler5 = MinMaxScaler(feature_range=(0,1),copy=True)
#["latitude","longitude","altitude"]

trayectorias_df[["latitude"]] = scaler1.fit_transform(trayectorias_df[["latitude"]])
trayectorias_df[["longitude"]] = scaler2.fit_transform(trayectorias_df[["longitude"]])
trayectorias_df[["altitude"]] = scaler3.fit_transform(trayectorias_df[["altitude"]])
trayectorias_df[["hav_distance"]] = scaler4.fit_transform(trayectorias_df[["hav_distance"]])
trayectorias_df[["speed"]] = scaler5.fit_transform(trayectorias_df[["speed"]])


In [8]:
trayectorias_df['id'] = trayectorias_df['fpId'].copy()

# Cambiamos de granularidad en segundos (donde hay saltos en los timesteps: 5-10-14-19-26...)
# al bucket de 15 segundos en el que se encuentra el vector (que debería reducir mucho los huecos)
trayectorias_df['timestamp'] = (trayectorias_df['timestamp'] - trayectorias_df['timestamp'].min()).astype(int)
trayectorias_df['timestamp'] = trayectorias_df['timestamp']//15

# others
trayectorias_df['categorical_id'] = trayectorias_df.ruta
trayectorias_df = trayectorias_df[['id','timestamp','latitude','longitude','altitude','track','sector','hav_distance','speed','ruta']].copy() # 'categorical_id',
trayectorias_df

,id,timestamp,latitude,longitude,altitude,track,sector,hav_distance,speed,ruta
0,AT05414795,285,0.775603,0.229187,0.023952,0.725374,1,0.457593,0.165493,EGLL
1,AT05414795,286,0.775593,0.228781,0.041372,0.707107,1,0.457502,0.154930,EGLL
2,AT05414795,287,0.775588,0.228393,0.054981,0.713250,1,0.457419,0.160798,EGLL
3,AT05414795,288,0.775577,0.227851,0.068590,0.725374,1,0.457301,0.185446,EGLL
4,AT05414795,289,0.775509,0.227490,0.076756,0.777146,1,0.457176,0.197183,EGLL
...,...,...,...,...,...,...,...,...,...,...
2971103,AT05829591,1570562,0.196756,0.149677,0.070223,0.325568,1,0.002460,0.170188,LIRN
2971104,AT05829591,1570563,0.197128,0.149488,0.066413,0.325568,1,0.002095,0.174883,LIRN
2971105,AT05829591,1570564,0.197590,0.149252,0.062058,0.325568,1,0.001641,0.173709,LIRN
2971106,AT05829591,1570565,0.198036,0.149027,0.057703,0.325568,1,0.001206,0.178404,LIRN


In [9]:
trayectorias_df['sector'] = trayectorias_df['sector'].astype(str)

In [10]:
trayectorias_df = trayectorias_df.sort_values(by='timestamp')

### Para evitar valores de index duplicados tras concatenar los datasets
trayectorias_df = trayectorias_df.reset_index(drop=True)

In [11]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from pytorch_forecasting.data import NaNLabelEncoder

encoders = dict(
    id=NaNLabelEncoder().fit(trayectorias_df.id)
)

In [12]:
max_prediction_length = 10 # predict max 1 day ahead
max_encoder_length = 60 #using last 7 days

# Sets limit for training data (last index prior to max prediction sequence)
training_cutoff = trayectorias_df.iloc[-829825].timestamp
validation_cutoff = trayectorias_df.iloc[-446642].timestamp

training = TimeSeriesDataSet(
    trayectorias_df[lambda x: x.timestamp <= training_cutoff],
    time_idx="timestamp", # variable that contains the time index
    target=["latitude","longitude","altitude"],
    group_ids=["id"], # Groups used for later normalizing
    min_encoder_length=max_encoder_length,# // 2,  # keep encoder length long (as it is in the validation set)
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    static_categoricals=["id"], # they do not change along a same time index
    static_reals=[], # same as before but real values
    time_varying_unknown_categoricals=["sector","ruta"],
    time_varying_unknown_reals=["latitude","longitude","altitude","track","hav_distance","speed"],
    categorical_encoders = encoders,
    allow_missing_timesteps = True, 
    target_normalizer=MultiNormalizer([TorchNormalizer(),TorchNormalizer(),TorchNormalizer()]),
    add_relative_time_idx=True,
    add_target_scales=True, # to add the center and scale of the unnormalized timeseries as features  TODO change to false
    add_encoder_length=True, # adds decoder length to list of static real variables. 
)

/home/usuario/.local/lib/python3.10/site-packages/pytorch_forecasting/data/timeseries.py:1238: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 29 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__id': 'AT03075869'}, {'__group_id__id': 'AT03623655'}, {'__group_id__id': 'AT03649638'}, {'__group_id__id': 'AT05421246'}, {'__group_id__id': 'AT05510200'}, {'__group_id__id': 'AT05529128'}, {'__group_id__id': 'AT05652888'}, {'__group_id__id': 'AT05754053'}, {'__group_id__id': 'AT05775000'}, {'__group_id__id': 'AT05779361'}]
  warnings.warn(


In [13]:
validation = TimeSeriesDataSet.from_dataset(training,
    trayectorias_df[lambda x: x.timestamp <= validation_cutoff], 
    min_prediction_idx=training_cutoff+1,
                                           )
testing = TimeSeriesDataSet.from_dataset(training,
    trayectorias_df,
    min_prediction_idx=validation_cutoff+1,
                                           )
batch_size = 128  # set this between 32 to 128
train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=6)
val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size * 10, num_workers=6)
test_dataloader = testing.to_dataloader(train=False, batch_size=batch_size * 10, num_workers=6)

/home/usuario/.local/lib/python3.10/site-packages/pytorch_forecasting/data/timeseries.py:1238: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 2 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__id': 'AT04126877'}, {'__group_id__id': 'AT04770153'}]
  warnings.warn(
/home/usuario/.local/lib/python3.10/site-packages/pytorch_forecasting/data/timeseries.py:1238: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 5 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__id': 'AT04780584'}, {'__group_id__id': 'AT04964385'}, {'__group_id__id': 'AT05217841'}, {'__group_id__id': 'AT05325960'}, {'__group_id__id': 'AT05795555'}]
  warnings.warn(


In [14]:
for i in train_dataloader:
    for j in i:
        if type(j)==tuple:
            continue
        for k,v in j.items():
            if type(v)==list:
                print(v[0].shape)
            else:
                print(v.shape)
    break

torch.Size([128, 61, 3])
torch.Size([128, 61, 14])
torch.Size([128, 61])
torch.Size([128])
torch.Size([128, 10, 3])
torch.Size([128, 10, 14])
torch.Size([128, 10])
torch.Size([128])
torch.Size([128, 10])
torch.Size([128, 1])
torch.Size([128, 2])


In [16]:
from torchmetrics import MeanSquaredError
from torch import nn
early_stop_callback = EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=10, verbose=False, mode="min")
lr_logger = LearningRateMonitor()  # log the learning rate
logger = TensorBoardLogger("trayectorias_logs4")  # logging results to a tensorboard

trainer = pl.Trainer(
    max_epochs=120,
    #gpus=1, #1 for gpu
    weights_summary="top", #for printing a summary of the model parameters
    gradient_clip_val=0.1,
    limit_train_batches=320,  # training over batches of 320
    callbacks=[lr_logger, early_stop_callback],
    logger=logger,
)

tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.1,
    hidden_size=200,  # most important hyperparameter apart from learning rate
    attention_head_size=4,
    dropout=0.3,  # between 0.1 and 0.3 are good values
    hidden_continuous_size=64,
    output_size=[7,7,7],
    logging_metrics = nn.ModuleList([MeanSquaredError()]),
    reduce_on_plateau_patience=5,
)



GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/home/usuario/.local/lib/python3.10/site-packages/pytorch_lightning/utilities/parsing.py:261: UserWarning: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
  rank_zero_warn(
/home/usuario/.local/lib/python3.10/site-packages/pytorch_lightning/utilities/parsing.py:261: UserWarning: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
  rank_zero_warn(


In [20]:
trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)


GPU available: True, used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                               | Type                            | Params
----------------------------------------------------------------------------------------
0  | loss                               | MultiLoss                       | 0     
1  | logging_metrics                    | ModuleList                      | 0     
2  | input_embeddings                   | MultiEmbedding                  | 142 K 
3  | prescalers                         | ModuleDict                      | 1.4 K 
4  | static_variable_selection          | VariableSelectionNetwork        | 23.6 K
5  | encoder_variable_selection         | VariableSelectionNetwork        | 12.3 K
6  | decoder_variable_selection         | VariableSelectionNetwork        | 2.8 K 
7  | static_context_variable_selection  | GatedResidualNetw

Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

In [23]:
best_model_path = trainer.checkpoint_callback.best_model_path
print(best_model_path)
best_tft = TemporalFusionTransformer.load_from_checkpoint("trayectorias_logs4/lightning_logs/12TFTTrack_ah4_u200_lb60/checkpoints/best.ckpt")

In [24]:
score = {"MSE" : best_tft.logging_metrics[0]}
predictions = best_tft.predict(test_dataloader)

#primero escalado
tft_MSE = score["MSE"](predictions[0], torch.cat([y[0][0] for x, y in iter(test_dataloader)]))
print(f"TFT Model MSE latitude escalado: {tft_MSE:.10f}")

tft_MSE = score["MSE"](predictions[1], torch.cat([y[0][1] for x, y in iter(test_dataloader)]))
print(f"TFT Model MSE longitude escalado: {tft_MSE:.10f}")

tft_MSE = score["MSE"](predictions[2], torch.cat([y[0][2] for x, y in iter(test_dataloader)]))
print(f"TFT Model MSE altitude escalado: {tft_MSE:.10f}")



#luego desescalado
a1=scaler1.inverse_transform(predictions[0].numpy())
a2=scaler2.inverse_transform(predictions[1].numpy())
a3=scaler3.inverse_transform(predictions[2].numpy())

t1 = torch.from_numpy(a1)
t2 = torch.from_numpy(a2)
t3 = torch.from_numpy(a3)

pred_desescalado = [t1,t2,t3]


actuals1 = torch.cat([y[0][0] for x, y in iter(test_dataloader)])
b1 = scaler1.inverse_transform(actuals1.numpy())
c1 = torch.from_numpy(b1)

tft_MSE = score["MSE"](pred_desescalado[0], c1)
print(f"TFT Model MSE latitude: {tft_MSE:.10f}")

actuals2 = torch.cat([y[0][1] for x, y in iter(test_dataloader)])
b2 = scaler2.inverse_transform(actuals2.numpy())
c2 = torch.from_numpy(b2)
tft_MSE = score["MSE"](pred_desescalado[1], c2)
print(f"TFT Model MSE longitude: {tft_MSE:.10f}")

actuals3 = torch.cat([y[0][2] for x, y in iter(test_dataloader)])
b3 = scaler3.inverse_transform(actuals3.numpy())
c3 = torch.from_numpy(b3)
tft_MSE = score["MSE"](pred_desescalado[2], c3)
print(f"TFT Model MSE altitude: {tft_MSE:.10f}")



TFT Model MSE latitude escalado: 0.0000150287
TFT Model MSE longitude escalado: 0.0000198863
TFT Model MSE altitude escalado: 0.0001827434
TFT Model MSE latitude: 0.0054568783
TFT Model MSE longitude: 0.0289767943
TFT Model MSE altitude: 385425.1250000000
